# 02 — Baseline Model: TF-IDF + Logistic Regression

This notebook trains and evaluates the baseline complaint classifier using
TF-IDF features and Logistic Regression.

**Theory:** TF-IDF (Term Frequency–Inverse Document Frequency) converts text
into weighted word-count vectors. Words that are frequent in one document but
rare across all documents get higher scores — giving the model discriminative
features without any deep learning.

Prerequisites: Run `01_eda.ipynb` or at least `python src/data_prep.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from src.baseline import train_baseline, predict_baseline, evaluate_baseline, get_top_features
from src.evaluate import evaluate_model, plot_confusion_matrix

sns.set_theme(style='whitegrid')
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data' / 'processed'
RESULTS_DIR = BASE_DIR / 'results'
MODELS_DIR = BASE_DIR / 'models'

print('Ready.')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

le_product = joblib.load(MODELS_DIR / 'label_encoder_product.joblib')
label_names = list(le_product.classes_)

X_train, y_train = train_df['narrative_clean'], train_df['product_encoded']
X_val,   y_val   = val_df['narrative_clean'],   val_df['product_encoded']
X_test,  y_test  = test_df['narrative_clean'],  test_df['product_encoded']

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Classes ({len(label_names)}): {label_names}')

## 2. Train — Primary Task (Product Classification)

In [ ]:
pipeline_product = train_baseline(
    X_train, y_train,
    task='product',
    max_features=50000,
    C=1.0,
)
print('Product classifier trained.')

## 3. Evaluate on Validation Set

In [ ]:
val_preds, val_probas = predict_baseline(pipeline_product, X_val)
val_metrics = evaluate_baseline(pipeline_product, X_val, y_val, label_names, task='product_val')

print(f"\nValidation Results:")
print(f"  Accuracy:    {val_metrics['accuracy']:.4f}")
print(f"  Macro F1:    {val_metrics['f1_macro']:.4f}")
print(f"  Weighted F1: {val_metrics['f1_weighted']:.4f}")
print(f"\n{val_metrics['classification_report']}")

## 4. Evaluate on Test Set (Final)

In [ ]:
test_preds, test_probas = predict_baseline(pipeline_product, X_test)
test_metrics = evaluate_model(
    y_test.values, test_preds, test_probas,
    label_names=label_names,
    model_name='baseline_product',
)

print(f"Test Results:")
print(f"  Accuracy:    {test_metrics['accuracy']:.4f}")
print(f"  Macro F1:    {test_metrics['f1_macro']:.4f}")
print(f"  Weighted F1: {test_metrics['f1_weighted']:.4f}")

In [ ]:
plot_confusion_matrix(
    y_test.values, test_preds,
    label_names=label_names,
    model_name='baseline_product',
    normalize=True,
)
plt.show() if False else None  # already saved; show inline too
from IPython.display import Image
Image(str(RESULTS_DIR / 'confusion_matrix_baseline_product.png'))

## 5. Top TF-IDF Features per Class

In [ ]:
top_features = get_top_features(pipeline_product, label_names, top_n=15)

# Display as table
feat_df = pd.DataFrame(top_features).T
feat_df.columns = [f'Feature #{i+1}' for i in range(feat_df.shape[1])]
print('Top 15 TF-IDF features per class:')
feat_df

In [ ]:
# Visualize top 10 features for each class
n_classes = len(label_names)
cols = 3
rows = (n_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

tfidf = pipeline_product.named_steps['tfidf']
clf   = pipeline_product.named_steps['clf']
feature_names = tfidf.get_feature_names_out()

for i, (label, ax) in enumerate(zip(label_names, axes)):
    coef = clf.coef_[i]
    top_idx = coef.argsort()[-10:][::-1]
    ax.barh(feature_names[top_idx][::-1], coef[top_idx][::-1],
            color='steelblue', alpha=0.8)
    ax.set_title(label[:30], fontsize=9, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top TF-IDF Features per Product Class (LR Coefficients)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'baseline_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Secondary Task — Issue Group Classification

In [ ]:
le_issue = joblib.load(MODELS_DIR / 'label_encoder_issue_group.joblib')
issue_label_names = list(le_issue.classes_)

X_train_iss, y_train_iss = train_df['narrative_clean'], train_df['issue_group_encoded']
X_test_iss,  y_test_iss  = test_df['narrative_clean'],  test_df['issue_group_encoded']

pipeline_issue = train_baseline(
    X_train_iss, y_train_iss,
    task='issue_group',
)

issue_preds, issue_probas = predict_baseline(pipeline_issue, X_test_iss)
issue_metrics = evaluate_model(
    y_test_iss.values, issue_preds, issue_probas,
    label_names=issue_label_names,
    model_name='baseline_issue_group',
)

print(f"Issue Group Test Results:")
print(f"  Accuracy:    {issue_metrics['accuracy']:.4f}")
print(f"  Macro F1:    {issue_metrics['f1_macro']:.4f}")
print(f"  Weighted F1: {issue_metrics['f1_weighted']:.4f}")
print(f"\n{issue_metrics['classification_report']}")

## 7. Summary

The baseline TF-IDF + Logistic Regression model establishes our performance floor.
Results are saved to `results/baseline_product_metrics.json` and `results/baseline_issue_group_metrics.json`.

Key observations:
- High TF-IDF weights reflect the most discriminative domain-specific terms.
- This model cannot capture word order, negation, or long-range context.
- The attention model and DistilBERT should improve on these limitations.